In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import accuracy_score
import os
from sklearn.preprocessing import LabelEncoder

In [2]:
# Load your dataset (assuming it's in CSV format)
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\cleaned data')
match_results = pd.read_csv('afl_match_results_cleaned.csv')
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\python scripts\\Catagorical prediction\\CatBoost')

# Split features and target
X = match_results.drop(columns=['match.homeTeam.name', 'match.awayTeam.name','venue.name','Margin','Result'])  # Replace 'target_column' with your target column name
# Initialize LabelEncoder
encoder = LabelEncoder()

# Fit and transform the target variable
y = encoder.fit_transform(match_results['Result'])

# Identify categorical features (you can list their indices or column names)
categorical_features = ['weather.weatherType']  # Replace with your actual categorical feature names

# Step 1: Split the data into train (80%) and test (20%) sets
train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

### Run at beginning of season

In [3]:
# import time
# start_time = time.time()

# # Step 2: Time series split for training and hyperparameter tuning on the train set
# tscv = TimeSeriesSplit(n_splits=5)  # Time series split with 5 splits

# # Step 3: Initialize CatBoostClassifier
# catboost_model = CatBoostClassifier(loss_function='MultiClass', eval_metric='Accuracy', random_seed=42)

# # Step 4: Hyperparameter tuning using GridSearchCV on the train set
# param_grid = {
#     'iterations': [100, 200, 500],  # Number of boosting iterations
#     'depth': [4, 6, 8],  # Tree depth
#     'learning_rate': [0.01, 0.1, 0.2],  # Learning rate
#     'l2_leaf_reg': [1, 3, 5],  # L2 regularization
#     'border_count': [32, 64, 128]  # Number of splits for numerical features
# }

# # Use GridSearchCV with time series split on the training data
# grid_search = GridSearchCV(estimator=catboost_model, param_grid=param_grid, 
#                            cv=tscv, scoring='accuracy', verbose=1, n_jobs=-1)

# # Step 5: Fit GridSearchCV on the train data
# grid_search.fit(X_train, y_train, cat_features=categorical_features)

# # Best parameters from GridSearchCV
# best_params = grid_search.best_params_
# print("Best hyperparameters:", best_params)
# print("--- %s seconds ---" % (time.time() - start_time))

### Continue programming

In [4]:
params = {
    'iterations': 200,  # Number of boosting iterations
    'depth': 4,  # Tree depth
    'learning_rate': 0.1,  # Learning rate
    'l2_leaf_reg': 1,  # L2 regularization
    'border_count': 64  # Number of splits for numerical features
}

# Step 6: Train the model with the best hyperparameters on the training set (using time series splits)
final_model = CatBoostClassifier(**params, loss_function='MultiClass', random_seed=42)
final_model.fit(X_train, y_train, cat_features=categorical_features)

# Step 7: Test the model on the test set
y_test_pred_probs = final_model.predict_proba(X_test)  # Get the probability for each class on the test set
y_test_pred_class = np.argmax(y_test_pred_probs, axis=1)  # Class with the highest probability

# Evaluate test accuracy
test_accuracy = accuracy_score(y_test, y_test_pred_class)
print(f"Test Accuracy: {test_accuracy:.4f}")

0:	learn: 1.5034605	total: 186ms	remaining: 37s
1:	learn: 1.4081517	total: 218ms	remaining: 21.5s
2:	learn: 1.3256606	total: 253ms	remaining: 16.6s
3:	learn: 1.2578044	total: 284ms	remaining: 13.9s
4:	learn: 1.1988735	total: 321ms	remaining: 12.5s
5:	learn: 1.1581431	total: 357ms	remaining: 11.5s
6:	learn: 1.1148790	total: 396ms	remaining: 10.9s
7:	learn: 1.0656404	total: 434ms	remaining: 10.4s
8:	learn: 1.0255755	total: 472ms	remaining: 10s
9:	learn: 0.9976678	total: 508ms	remaining: 9.65s
10:	learn: 0.9710896	total: 546ms	remaining: 9.38s
11:	learn: 0.9460754	total: 583ms	remaining: 9.14s
12:	learn: 0.9239273	total: 623ms	remaining: 8.96s
13:	learn: 0.9007979	total: 671ms	remaining: 8.91s
14:	learn: 0.8845790	total: 727ms	remaining: 8.96s
15:	learn: 0.8669754	total: 773ms	remaining: 8.89s
16:	learn: 0.8505188	total: 815ms	remaining: 8.77s
17:	learn: 0.8382053	total: 856ms	remaining: 8.65s
18:	learn: 0.8257351	total: 898ms	remaining: 8.55s
19:	learn: 0.8116970	total: 937ms	remaining: 

In [5]:
# Step 8: Train on the full dataset (after testing)
final_model.fit(X, y, cat_features=categorical_features, verbose=100)

0:	learn: 1.4955592	total: 28.4ms	remaining: 5.65s
100:	learn: 0.5239144	total: 3.8s	remaining: 3.73s
199:	learn: 0.3937118	total: 7.5s	remaining: 0us


In [6]:
final_model.save_model('final_model.cbm')
import pickle
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
with open('accuracy.pkl', 'wb') as f:
    pickle.dump(test_accuracy, f)